In [2]:
import tqdm
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from transformers import Mistral3ForConditionalGeneration, FineGrainedFP8Config





In [ ]:
model_id = "mistralai/Ministral-3-3B-Instruct-2512"

prompt_system = "You are a helpful IA trying to recommand books based on a query by a user. " \
"Your answer includes 3 books, with a short summary of each and their type."

# prompt_system = "You answer as best you can to the question you are asked."

In [17]:

prompt_user = "What will the weather be tomorrow ?"

chat = [
    {"role" : "system", "content" : prompt_system},
    {"role" : "user", "content" : prompt_user}
]


In [5]:
# model_id = "mistralai/Ministral-3-3B-Instruct-2512"
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

# model = AutoModelForCausalLM.from_pretrained(model)
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map="auto", dtype=torch.bfloat16)

Loading weights: 100%|██████████| 458/458 [00:08<00:00, 51.15it/s]
[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [18]:

   
# Quick evaluation runs
tokenized_prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device) 
# inputs = tokenizer(tokenized_prompt, return_tensors="pt")
outputs = model.generate(**tokenized_prompt, max_new_tokens=500)
print(tokenizer.decode(outputs[0]))

[transformers] Both `max_new_tokens` (=500) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[SYSTEM_PROMPT]You answer as best you can to the question you are asked.[/SYSTEM_PROMPT][INST]What will the weather be tomorrow ?[/INST]I can't provide real-time weather updates, but you can check the weather for tomorrow using reliable sources like:

- **Weather apps** (e.g., AccuWeather, The Weather Channel, Weather.com)
- **Weather websites** (e.g., [Weather Underground](https://www.wunderground.com/), [NOAA](https://www.weather.gov/))
- **Local news or weather forecasts** from your country’s meteorological service

For example, if you're in the U.S., you could check [NOAA’s National Weather Service](https://www.weather.gov/) for your specific location.</s>
